# Smart Traffic — Congestion Classifier **v2** (multi-dataset + YOLO pivot)

This is the upgraded training notebook. The v1 notebook trained the 4-class congestion
model on a **single** source (Norway traffic cameras). It worked, but accuracy plateaued.
This version attacks the three real causes and gives you a head-to-head comparison to put
in the report.

## Why v1's accuracy stalled (the honest diagnosis)
1. **One camera domain.** Every image is a fixed Norwegian road-cam from a similar angle.
   The net learns *that* viewpoint, not "traffic", so it generalises poorly to other
   positions — exactly what you noticed.
2. **Subjective adjacent classes.** `low` vs `high`, `high` vs `jam` are fuzzy human calls;
   the labels themselves are noisy, which caps any classifier.
3. **A blunt training recipe.** v1 fine-tuned the *whole* pretrained net at `lr=1e-3` from
   epoch 1 (wrecks ImageNet features early) and saved the *last* epoch, not the best.

## What v2 does
- **Track A — better classifier:** trains on **multiple datasets** (Norway + Traffic-Net's
  web-sourced sparse/dense images → viewpoint diversity, **plus an optional ImageFolder
  slot for your own new-position captures**), with a proper **two-phase transfer-learning**
  recipe (freeze→fine-tune), label smoothing, class-balanced sampling, stronger
  augmentation, cosine LR, and **save-best-on-val**. Outputs the exact same `traffic.pt`
  the service already loads — no code change downstream.
- **Track B — the pivot you asked for:** a COCO-pretrained **YOLO that literally detects
  cars**, counts vehicles per image, and maps the count → `empty/low/high/jam`. **Zero
  training**, viewpoint-robust, and explainable in a defense. Evaluated on the *same*
  validation set so you get an apples-to-apples number.

Run both, look at the two confusion matrices, and ship whichever wins (or ensemble them).

**Before running:** *Runtime → Change runtime type → GPU*. Then *Runtime → Run all*.
Self-contained: datasets are pulled onto Colab (nothing touches your PC, nothing goes to
GitHub); only the small `traffic.pt` comes back.

### Label harmonisation (each source declares its own mapping)
| source | dataset class | → our class | weight |
|---|---|---|---|
| Norway | `no-traffic` | `empty` | 1 |
| Norway | `low-traffic` | `low` | 2 |
| Norway | `medium-traffic` | `high` | 3 |
| Norway | `high-traffic` | `jam` | 4 |
| Traffic-Net | `sparse_traffic` | `low` | 2 |
| Traffic-Net | `dense_traffic` | `jam` | 4 |

> Traffic-Net only covers two congestion levels (its `accident`/`fire` classes are out of
> scope and dropped). It enriches `low` and `jam` with diverse viewpoints; Norway supplies
> all four levels. Toggle `USE_TRAFFICNET` off to A/B test whether it actually helps —
> the notebook reports a **Norway-only** validation number so the comparison is fair.
>
> **Add new viewpoints (3rd+ source):** set `USE_IMAGEFOLDER = True` and drop images into
> `data/extra/{empty,low,high,jam}/` (or `data/extra/{train,val}/{empty,low,high,jam}/`).
> Folder names are the taxonomy, so no mapping is needed — this is the slot for your own
> captured intersection frames or any Kaggle/Roboflow download.

### 1. Install
Colab already ships torch + torchvision (CUDA). We add `datasets` (Hugging Face) and
`ultralytics` (YOLO for Track B).

In [ ]:
!pip -q install datasets ultralytics

### 2. Configuration
Toggle datasets and tracks here. The taxonomy + weights MUST match `app/config.py` and
`apps/server/src/types.ts` — don't change them.

In [ ]:
import os, json, time, random, glob, zipfile, urllib.request, itertools
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from datasets import load_dataset
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device != "cuda":
    print("WARNING: no GPU -> Runtime > Change runtime type > GPU for a fast run.")

# --- our taxonomy (must match the service) ---
CLASSES = ["empty", "low", "high", "jam"]
WEIGHTS = {"empty": 1, "low": 2, "high": 3, "jam": 4}
W_MAX = 4
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

# --- datasets to mix in (each declares its own label map below) ---
USE_NORWAY      = True   # 4-level Norway traffic-cam dataset (Hugging Face)
USE_TRAFFICNET  = True   # web-sourced sparse/dense images -> viewpoint diversity
USE_IMAGEFOLDER = False  # YOUR extra images / new camera positions (see below)
IMAGEFOLDER_DIR = "data/extra"   # put images in {empty,low,high,jam}/  (optionally under train/ val/)

# --- what to run ---
RUN_TRACK_A    = True   # train the EfficientNet classifier
RUN_TRACK_B    = True   # YOLO vehicle-count baseline (no training)

# --- classifier (Track A) hyper-parameters ---
IMG_SIZE     = 224
BATCH        = 32
HEAD_EPOCHS  = 3        # phase 1: train the new head only (backbone frozen)
FT_EPOCHS    = 10       # phase 2: fine-tune the whole network at a low LR
ARCH         = "efficientnet"
LABEL_SMOOTH = 0.1
OUT          = "traffic.pt"

random.seed(0); torch.manual_seed(0)

### 3. Load + harmonise the datasets
We build one unified list of samples. Each sample is `(loader -> PIL image, class index,
source)` so multiple datasets plug in through their own label maps, and we can still slice
by source for a fair comparison.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]; IMAGENET_STD = [0.229, 0.224, 0.225]

train_samples, val_samples = [], []
source_counts = {}

def _add(split_list, get_img, our_class, source):
    split_list.append((get_img, CLASS_TO_IDX[our_class], source))
    source_counts.setdefault(source, [0, 0, 0, 0])[CLASS_TO_IDX[our_class]] += 1

# ---------- source 1: Norway traffic-camera (HF, 4 ordinal levels) ----------
NORWAY_MAP = {"no-traffic": "empty", "low-traffic": "low",
              "medium-traffic": "high", "high-traffic": "jam"}
if USE_NORWAY:
    raw = load_dataset("ilsilfverskiold/traffic-camera-norway-images")
    int2str = raw["train"].features["label"].int2str
    def hf_getter(split, i):
        return lambda: raw[split][i]["image"].convert("RGB")
    for split, dst in [("train", train_samples), ("validation", val_samples)]:
        labels = raw[split]["label"]
        for i, lab in enumerate(labels):
            _add(dst, hf_getter(split, i), NORWAY_MAP[int2str(lab)], "norway")
    print("loaded Norway:", {k: raw[k].num_rows for k in raw})

# ---------- source 2: Traffic-Net (web images: sparse/dense -> viewpoint diversity) ----------
TRAFFICNET_MAP = {"sparse_traffic": "low", "dense_traffic": "jam"}  # accident/fire dropped
if USE_TRAFFICNET:
    url = "https://github.com/OlafenwaMoses/Traffic-Net/releases/download/1.0/trafficnet_dataset_v1.zip"
    if not glob.glob("**/sparse_traffic", recursive=True):
        print("downloading Traffic-Net (50 MB)...")
        urllib.request.urlretrieve(url, "trafficnet.zip")
        with zipfile.ZipFile("trafficnet.zip") as z:
            z.extractall(".")
    def file_getter(path):
        return lambda: Image.open(path).convert("RGB")
    for ds_class, our_class in TRAFFICNET_MAP.items():
        for folder in glob.glob(f"**/{ds_class}", recursive=True):
            fl = folder.replace("\\", "/").lower()
            dst = train_samples if "/train" in fl else (val_samples if "/test" in fl else None)
            if dst is None:
                continue
            for p in glob.glob(folder + "/*"):
                if p.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                    _add(dst, file_getter(p), our_class, "trafficnet")

# ---------- source 3: YOUR extra ImageFolder (new camera positions) ----------
# The clean way to add new viewpoints: drop images into
#   IMAGEFOLDER_DIR/{empty,low,high,jam}/*.jpg      (all go to train), or
#   IMAGEFOLDER_DIR/{train,val}/{empty,low,high,jam}/*.jpg   (explicit split).
# Folder names ARE our taxonomy, so no mapping needed. This is also how you train
# on your own captured intersection images, or any Kaggle/Roboflow download.
if USE_IMAGEFOLDER:
    def folder_getter(path):
        return lambda: Image.open(path).convert("RGB")
    added = 0
    for our_class in CLASSES:
        for folder in glob.glob(f"{IMAGEFOLDER_DIR}/**/{our_class}", recursive=True):
            fl = folder.replace("\\", "/").lower()
            dst = val_samples if ("/val" in fl or "/test" in fl) else train_samples
            for p in glob.glob(folder + "/*"):
                if p.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
                    _add(dst, folder_getter(p), our_class, "imagefolder"); added += 1
    print(f"loaded ImageFolder extra: {added} images from {IMAGEFOLDER_DIR}/")

print("\nper-source class counts [empty, low, high, jam]:")
for s, c in source_counts.items():
    print(f"  {s:11s} {c}  (total {sum(c)})")
print(f"\nTRAIN total {len(train_samples)} | VAL total {len(val_samples)}")
assert train_samples, "no training data -- enable at least one dataset"


### 4. Torch datasets, augmentation, class-balanced sampler
Stronger-but-sane augmentation (no vertical flips — roads have a fixed orientation) plus a
`WeightedRandomSampler` so the heavy class imbalance doesn't drown the rare classes.

In [ ]:
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.65, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandAugment(num_ops=2, magnitude=7),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.1),
])
eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class SampleDS(Dataset):
    def __init__(self, samples, tf): self.samples, self.tf = samples, tf
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        get_img, y, _ = self.samples[i]
        return self.tf(get_img()), y

counts = [0, 0, 0, 0]
for _, y, _ in train_samples: counts[y] += 1
print("train class counts:", dict(zip(CLASSES, counts)))
class_w = [1.0 / c if c else 0.0 for c in counts]
sample_w = [class_w[y] for _, y, _ in train_samples]
sampler = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)

train_loader = DataLoader(SampleDS(train_samples, train_tf), batch_size=BATCH,
                          sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(SampleDS(val_samples, eval_tf), batch_size=BATCH,
                        shuffle=False, num_workers=2, pin_memory=True)

### 5. Model — EfficientNetV2-S, ImageNet-pretrained, with freeze/unfreeze
Same architecture the service rebuilds when it loads `traffic.pt`, so the checkpoint drops
straight in.

In [ ]:
def build_model(num_classes=4):
    net = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)
    net.classifier[-1] = nn.Linear(net.classifier[-1].in_features, num_classes)
    return net

def set_backbone_frozen(net, frozen):
    for n, p in net.named_parameters():
        if not n.startswith("classifier"):
            p.requires_grad = not frozen

model = build_model(len(CLASSES)).to(device)
print(f"EfficientNetV2-S: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")

### 6. Evaluation — mirrors `app/scoring.py`
Accuracy, **mean weighted error** (the ordinal penalty: confusing `empty`↔`jam` costs more
than `low`↔`high`), mean score, confusion matrix, and per-class P/R/F1.

In [ ]:
@torch.no_grad()
def evaluate(net, loader):
    net.eval()
    conf = {t: {p: 0 for p in CLASSES} for t in CLASSES}
    n = correct = 0; sum_err = sum_score = 0.0
    for x, y in loader:
        prob = torch.softmax(net(x.to(device)), 1).cpu()
        cfd, pred = prob.max(1)
        for t_idx, p_idx, cf in zip(y.tolist(), pred.tolist(), cfd.tolist()):
            t, p = CLASSES[t_idx], CLASSES[p_idx]
            conf[t][p] += 1; n += 1; correct += int(t == p)
            e = abs(WEIGHTS[p] - WEIGHTS[t]) / W_MAX
            sum_err += e; sum_score += cf * (1 - e)
    return _finalize(conf, n, correct, sum_err, sum_score)

def _finalize(conf, n, correct, sum_err, sum_score):
    per = {}
    for c in CLASSES:
        tp = conf[c][c]
        fn = sum(conf[c].values()) - tp
        fp = sum(conf[t][c] for t in CLASSES) - tp
        P = tp / (tp + fp) if (tp + fp) else 0.0
        R = tp / (tp + fn) if (tp + fn) else 0.0
        F = 2 * P * R / (P + R) if (P + R) else 0.0
        per[c] = {"precision": round(P, 4), "recall": round(R, 4),
                  "f1": round(F, 4), "support": tp + fn}
    return {"accuracy": correct / max(n, 1), "mean_weighted_error": sum_err / max(n, 1),
            "mean_score": sum_score / max(n, 1), "confusion": conf, "per_class": per, "n": n}

def report(m, title):
    print(f"\n=== {title} ===")
    print(f"accuracy {m['accuracy']:.4f} | mean weighted error {m['mean_weighted_error']:.4f} "
          f"| mean score {m['mean_score']:.4f} | n={m['n']}")
    for c in CLASSES:
        x = m["per_class"][c]
        print(f"  {c:6s} P={x['precision']:.3f} R={x['recall']:.3f} F1={x['f1']:.3f} (n={x['support']})")
    print("  confusion (rows=true, cols=pred): " + "  ".join(f"{c:>5s}" for c in CLASSES))
    for t in CLASSES:
        print(f"  {t:>6s} " + " ".join(f"{m['confusion'][t][p]:5d}" for p in CLASSES))

### 7. Track A — two-phase transfer learning
**Phase 1** freezes the pretrained backbone and trains only the new head (so the random
head doesn't send garbage gradients into good features). **Phase 2** unfreezes everything
and fine-tunes at a low LR with a cosine schedule. We keep the **best** val checkpoint, not
the last.

In [ ]:
if RUN_TRACK_A:
    crit = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
    scaler = torch.cuda.amp.GradScaler(enabled=device == "cuda")
    best = {"acc": -1.0, "state": None}

    def run_phase(net, epochs, lr, tag):
        opt = torch.optim.AdamW([p for p in net.parameters() if p.requires_grad],
                                lr=lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs, 1))
        for e in range(1, epochs + 1):
            net.train(); run = 0.0; t0 = time.time()
            for x, y in train_loader:
                x, y = x.to(device), y.to(device)
                opt.zero_grad()
                with torch.autocast(device_type=device, enabled=device == "cuda"):
                    loss = crit(net(x), y)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                run += loss.item()
            sched.step()
            m = evaluate(net, val_loader)
            flag = ""
            if m["accuracy"] > best["acc"]:
                best["acc"] = m["accuracy"]
                best["state"] = {k: v.cpu().clone() for k, v in net.state_dict().items()}
                flag = "  *best"
            print(f"[{tag}] epoch {e:2d}/{epochs}  loss={run/len(train_loader):.3f}  "
                  f"val_acc={m['accuracy']:.4f}  mwe={m['mean_weighted_error']:.4f}  "
                  f"({time.time()-t0:.0f}s){flag}")

    set_backbone_frozen(model, True);  run_phase(model, HEAD_EPOCHS, 1e-3, "head")
    set_backbone_frozen(model, False); run_phase(model, FT_EPOCHS, 1e-4, "finetune")
    model.load_state_dict(best["state"])
    print(f"\nrestored best checkpoint (val_acc={best['acc']:.4f})")

### 8. Track A — final report
Reported twice: on the **combined** validation set, and on **Norway-only** (directly
comparable to the old single-source run, so you can prove whether v2 actually moved the
needle).

In [ ]:
if RUN_TRACK_A:
    report(evaluate(model, val_loader), "Track A - combined validation")
    nv_samples = [s for s in val_samples if s[2] == "norway"]
    if nv_samples:
        nv = DataLoader(SampleDS(nv_samples, eval_tf), batch_size=BATCH, num_workers=2)
        mA = evaluate(model, nv)
        report(mA, "Track A - Norway-only validation (comparable to the old run)")
        json.dump(mA, open("metrics_trackA.json", "w"), indent=2)

### 9. Save the checkpoint (exact service contract) + ONNX

In [ ]:
if RUN_TRACK_A:
    torch.save({"state_dict": model.state_dict(), "arch": ARCH, "classes": CLASSES}, OUT)
    print(f"saved {OUT}  ({os.path.getsize(OUT)/1e6:.1f} MB)  arch={ARCH}  classes={CLASSES}")
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
    torch.onnx.export(model.eval(), dummy, "traffic.onnx",
                      input_names=["image"], output_names=["logits"],
                      dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}},
                      opset_version=17)
    print("exported traffic.onnx")

In [ ]:
# download the trained weights to your PC -> drop into services/ml/models/traffic.pt
if RUN_TRACK_A:
    from google.colab import files
    files.download("traffic.pt")

### 10. Track B — the pivot: count cars with a pretrained YOLO
A COCO-pretrained YOLO already detects `car`/`motorcycle`/`bus`/`truck` with no training
and is robust to camera angle (it was trained on millions of varied images). We count
vehicles per image, then calibrate three count thresholds on the Norway **train** labels to
map a count → `empty/low/high/jam`, and evaluate on Norway **val**.

This is the literal "detect cars" approach, it directly fixes the *different-positions*
weakness, and it's easy to justify in a defense: *we count vehicles and threshold density.*

In [ ]:
if RUN_TRACK_B:
    from ultralytics import YOLO
    yolo = YOLO("yolo11n.pt")                       # COCO-pretrained, auto-downloads
    VEHICLES = {2, 3, 5, 7}                          # COCO ids: car, motorcycle, bus, truck

    def count_vehicles(pil):
        r = yolo.predict(pil, verbose=False, conf=0.25, imgsz=640)[0]
        if r.boxes is None:
            return 0
        return sum(1 for i in r.boxes.cls.int().tolist() if i in VEHICLES)

In [ ]:
if RUN_TRACK_B:
    norway_train = [s for s in train_samples if s[2] == "norway"]
    random.shuffle(norway_train)
    CALIB_N = 1500                                   # lower this if you're short on time
    cal = norway_train[:CALIB_N]
    nv_samples = [s for s in val_samples if s[2] == "norway"]
    print(f"YOLO counting on {len(cal)} calibration + {len(nv_samples)} validation images...")

    t0 = time.time()
    cal_counts = [(count_vehicles(g()), y) for g, y, _ in cal]
    val_counts = [(count_vehicles(g()), y) for g, y, _ in nv_samples]
    print(f"counted in {time.time()-t0:.0f}s")

    def to_class(c, t):                              # t = (t1, t2, t3) ascending
        return 0 if c <= t[0] else 1 if c <= t[1] else 2 if c <= t[2] else 3
    def acc(counts, t):
        return sum(to_class(c, t) == y for c, y in counts) / len(counts)

    # grid-search ascending integer thresholds on the calibration set
    best = (-1.0, None)
    for t in itertools.combinations(range(0, 31), 3):
        a = acc(cal_counts, t)
        if a > best[0]:
            best = (a, t)
    T = best[1]
    print(f"calibrated: empty<= {T[0]} <low<= {T[1]} <high<= {T[2]} <jam   "
          f"(calibration acc {best[0]:.4f})")
    # plug these into the ML service so the LIVE stream's count-based label matches:
    print(f'-> set on the ML service:  COUNT_THRESHOLDS="{T[0]},{T[1]},{T[2]}"')

    conf = {t: {p: 0 for p in CLASSES} for t in CLASSES}
    n = correct = 0; sum_err = 0.0
    for c, y in val_counts:
        p_idx = to_class(c, T); t, p = CLASSES[y], CLASSES[p_idx]
        conf[t][p] += 1; n += 1; correct += int(t == p)
        sum_err += abs(WEIGHTS[p] - WEIGHTS[t]) / W_MAX
    mB = _finalize(conf, n, correct, sum_err, 0.0)   # no softmax confidence -> mean_score N/A
    report(mB, "Track B - YOLO vehicle-count (Norway validation)")
    json.dump({"thresholds": T, "metrics": mB}, open("metrics_trackB.json", "w"), indent=2)

### 11. Verdict — what to ship

Compare the two **Norway-only** validation blocks (same images, fair fight):

- **Track A wins** → you already have `traffic.pt`; drop it into
  `services/ml/models/traffic.pt`, restart the service, done. Report the v1→v2 accuracy
  lift and credit the multi-dataset + two-phase recipe.
- **Track B wins** (common when viewpoints vary) → the live service already supports it:
  set `COUNT_THRESHOLDS="t0,t1,t2"` (printed by the calibration cell) on the ML service and
  the **live stream's** vehicle-count panel will use your calibrated mapping. The signal
  engine, scoring, and dashboard are unchanged because they only consume a label. Put the
  thresholds and the YOLO confusion matrix in the report.
- **Close call** → ensemble: take the classifier's class, but override to Track B's when
  the classifier is low-confidence. Easy win, easy to justify.

### For the report (all honest, all defensible)
- v1 vs v2 accuracy on the identical Norway validation split.
- Both confusion matrices — show *where* errors live (almost always adjacent classes:
  `low`↔`high`, `high`↔`jam`). That's the subjective-boundary problem, not a bug.
- **Mean weighted error** matters more than raw accuracy here: an `empty`→`jam` mistake is
  the one that actually breaks a traffic signal; adjacent slips barely change the green
  split. Lead with this metric.
- State plainly that the 4-class boundaries are inherently fuzzy, and that the YOLO-count
  approach is the more *objective* framing — a strong point for the defense.